# 🧪 Comprehensive Silver Layer Testing & Audit
Before moving to the **Gold Layer** (Aggregations & Business Intelligence), we must absolutely guarantee that the **Silver Layer** data is clean, typed correctly, and free of anomalies. 

This notebook runs **5 specific groups of tests** to validate the pipeline.

In [11]:
import os
import sys
import duckdb
import pandas as pd
from dotenv import load_dotenv

sys.path.append(os.path.abspath('./'))
from src.helper_files.database import DBConnection

load_dotenv()
SILVER_ZONE = os.getenv('SILVER_ZONE')

con = duckdb.connect(':memory:')
silver_glob = os.path.join(SILVER_ZONE, "*.parquet").replace("\\", "/")
print(f"Targeting Silver Zone: {SILVER_ZONE}")

Targeting Silver Zone: C:/Omnijourney_Kofking_github/data/silver


## TEST GROUP 1: Pipeline Processing Analytics
**What this tests:** Did Airflow successfully process the files? Were any files skipped or failed? This ensures we aren't missing any data before we aggregate it in Gold.

In [12]:
with DBConnection() as conn:
    query = """
    SELECT layer, status, COUNT(*) as file_count, SUM(row_count) as total_rows
    FROM file_lineage
    GROUP BY layer, status
    ORDER BY layer ASC, status DESC;
    """
    df_lineage = pd.read_sql(query, conn)

display(df_lineage)

C:\Users\FOZCOMPUTERSDXB\AppData\Local\Temp\ipykernel_7224\770296103.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_lineage = pd.read_sql(query, conn)


,layer,status,file_count,total_rows
0,BRONZE,SUCCESS,12,2034534.0
1,SILVER,SUCCESS,12,2034534.0


## TEST GROUP 2: Schema & Data Type Enforcement
**What this tests:** Are our numbers actually `BIGINT` and `DOUBLE`? If prices are stored as text (`VARCHAR`), Gold layer math (like `SUM()` or `AVG()`) will instantly crash.

In [13]:
df_schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{silver_glob}')").df()
display(df_schema[['column_name', 'column_type']])

,column_name,column_type
0,property_id,BIGINT
1,location_id,INTEGER
2,url_hash,VARCHAR
3,page_url,VARCHAR
4,property_type,VARCHAR
5,price,BIGINT
6,price_per_marla,DOUBLE
7,location,VARCHAR
8,city,VARCHAR
9,province_name,VARCHAR


## TEST GROUP 3: Data Completeness (Null Checks)
**What this tests:** Did the transformation accidentally delete data? We need to know exactly how many rows are missing prices, locations, or dates so we can filter them out or handle them in Gold.

In [14]:
query = f"""
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) as missing_prices,
    SUM(CASE WHEN area_marla IS NULL THEN 1 ELSE 0 END) as missing_areas,
    SUM(CASE WHEN latitude IS NULL THEN 1 ELSE 0 END) as missing_latitudes,
    SUM(CASE WHEN date_added IS NULL THEN 1 ELSE 0 END) as missing_dates
FROM read_parquet('{silver_glob}')
"""
df_quality = con.execute(query).df()
display(df_quality)

,total_rows,missing_prices,missing_areas,missing_latitudes,missing_dates
0,2034534,0.0,0.0,121.0,0.0


## TEST GROUP 4: Business Logic & Outlier Validation
**What this tests:** In `silver_transform.py`, we explicitly capped prices at 500 Million and standardized area to Marla. We must test MIN, MAX, and AVG to prove this logic actually worked and no trillion-dollar anomalies leaked through.

In [15]:
query = f"""
SELECT 
    MIN(price) as min_price,
    MAX(price) as max_price,
    AVG(price) as avg_price,
    MIN(area_marla) as min_area,
    MAX(area_marla) as max_area
FROM read_parquet('{silver_glob}')
"""
df_outliers = con.execute(query).df()
display(df_outliers)

,min_price,max_price,avg_price,min_area,max_area
0,0,500000000,1.766896e+07,0.0,16000.0


## TEST GROUP 5: Categorical Analysis (For Gold Layer Dimensions)
**What this tests:** To build Gold layer aggregations (like 'Average Price per City'), we need to know what unique categories exist in the data. This test lists the unique property types and purposes.

In [16]:
query_types = f"SELECT property_type, COUNT(*) as count FROM read_parquet('{silver_glob}') GROUP BY property_type ORDER BY count DESC"
query_cities = f"SELECT city, COUNT(*) as count FROM read_parquet('{silver_glob}') GROUP BY city ORDER BY count DESC LIMIT 5"

print("🏠 Property Types:")
display(con.execute(query_types).df())

print("\n🏙️ Top 5 Cities:")
display(con.execute(query_cities).df())

🏠 Property Types:


,property_type,count
0,House,1273321
1,Flat,461647
2,Upper Portion,166608
3,Lower Portion,111892
4,Room,8281
5,Farm House,7967
6,Penthouse,4818



🏙️ Top 5 Cities:


,city,count
0,Karachi,731035
1,Lahore,500155
2,Islamabad,451783
3,Rawalpindi,253671
4,Faisalabad,97890
